# a 数据清洗与图表分析\n\n读取 `a/国债ETF数据/3秒快照` 下的 ETF 3 秒快照数据。当前清洗口径：删除导出残留索引列，统一时间戳，保留所有交易阶段，把价格字段里的无效 `0` 先转成 `NaN`，暂时不填补也不删除。

In [ ]:
from pathlib import Path\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nwarnings.filterwarnings('ignore')\nplt.rcParams['figure.figsize'] = (14, 6)\nplt.rcParams['axes.grid'] = True\nplt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']\nplt.rcParams['axes.unicode_minus'] = False\n\ncwd = Path.cwd()\nPROJECT_ROOT = cwd.parent if cwd.name.lower() == 'notebooks' else cwd\nA_ROOT = PROJECT_ROOT / 'a' / '国债ETF数据' / '3秒快照'\nraw_files = sorted(A_ROOT.rglob('*snapshot.csv'))\npd.DataFrame({'file': [str(p.relative_to(PROJECT_ROOT)) for p in raw_files], 'size_mb': [round(p.stat().st_size/1024/1024, 2) for p in raw_files]})

## 读取与轻量清洗\n\n注意：`volume/amount/num_trades` 的 0 暂时保留；价格、盘口价、IOPV 的 0 先标成 NaN。

In [ ]:
PRICE_COLS = ['pre_close','last','open','high','low','close','high_limited','low_limited','ask_price1','bid_price1','ask_price2','bid_price2','ask_price3','bid_price3','ask_price4','bid_price4','ask_price5','bid_price5','iopv']\nVOLUME_COLS = ['volume','amount','num_trades','ask_volume1','bid_volume1','ask_volume2','bid_volume2','ask_volume3','bid_volume3','ask_volume4','bid_volume4','ask_volume5','bid_volume5']\n\ndef read_one(path):\n    df = pd.read_csv(path, low_memory=False)\n    df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed')]\n    df = df.loc[:, df.columns.astype(str).str.strip() != '']\n    df.columns = df.columns.astype(str).str.strip()\n    df['source_file'] = path.name\n    df['symbol_dir'] = path.parent.name\n    df['code'] = df['code'].astype('string').str.strip()\n    df['trading_phase_code'] = df['trading_phase_code'].astype('string').str.strip()\n    df['trade_time'] = pd.to_datetime(df['trade_time'], errors='coerce')\n    for c in PRICE_COLS + VOLUME_COLS:\n        if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')\n    cols = [c for c in PRICE_COLS if c in df.columns]\n    df[cols] = df[cols].replace(0, np.nan)\n    df['date'] = df['trade_time'].dt.date\n    df['is_continuous'] = df['trading_phase_code'].eq('T111')\n    return df\n\ndef read_symbol(symbol):\n    fs = [p for p in raw_files if p.parent.name == symbol]\n    df = pd.concat([read_one(p) for p in fs], ignore_index=True)\n    return df.sort_values(['code','trade_time','source_file']).reset_index(drop=True)\n\netf_all = pd.concat([read_symbol('511090'), read_symbol('511130')], ignore_index=True)\netf_all.shape, etf_all.head()

## 质量检查

In [ ]:
quality = etf_all.groupby('code').agg(rows=('code','size'), start=('trade_time','min'), end=('trade_time','max'), last_nan_rate=('last', lambda s: s.isna().mean()), bid1_nan_rate=('bid_price1', lambda s: s.isna().mean()), ask1_nan_rate=('ask_price1', lambda s: s.isna().mean()))\ndup = etf_all.duplicated(['code','trade_time']).groupby(etf_all['code']).sum().rename('duplicate_code_time')\ndisplay(quality.join(dup))\nphase_table = pd.crosstab(etf_all['code'], etf_all['trading_phase_code'])\ndisplay(phase_table)\nchecks = etf_all.assign(bid_gt_ask=etf_all['bid_price1'].notna() & etf_all['ask_price1'].notna() & (etf_all['bid_price1'] > etf_all['ask_price1']), last_outside_limit=etf_all['last'].notna() & etf_all['high_limited'].notna() & etf_all['low_limited'].notna() & ((etf_all['last'] > etf_all['high_limited']) | (etf_all['last'] < etf_all['low_limited'])))\nchecks.groupby('code')[['bid_gt_ask','last_outside_limit']].sum()

## 日频聚合数据\n\n图表默认基于连续交易阶段 `T111`。原始 NaN 不填补。

In [ ]:
continuous = etf_all[etf_all['is_continuous']].copy()\ndaily = (continuous.set_index('trade_time').groupby('code').resample('1D').agg(open=('last','first'), high=('last','max'), low=('last','min'), close=('last','last'), volume=('volume','last'), amount=('amount','last'), rows=('last','size'), last_nan_rate=('last', lambda s: s.isna().mean())).reset_index())\ndaily = daily.dropna(subset=['open','high','low','close'], how='all')\ndaily['return'] = daily.groupby('code')['close'].pct_change()\ndaily['cum_return'] = (1 + daily['return'].fillna(0)).groupby(daily['code']).cumprod() - 1\ndaily.head()

## 图 1-4：折线图、K 线图、柱状图、多折线图

In [ ]:
# 1 折线图\nfor code, g in daily.groupby('code'):\n    plt.plot(g['trade_time'], g['close'], marker='o', linewidth=1.6, label=code)\nplt.title('ETF 日收盘价趋势'); plt.xlabel('日期'); plt.ylabel('收盘价'); plt.legend(); plt.show()\n\n# 2 K 线图\ndef plot_candles(df, code, last_n=80):\n    g = df[df['code'].eq(code)].dropna(subset=['open','high','low','close']).tail(last_n).copy()\n    x = np.arange(len(g)); fig, ax = plt.subplots(figsize=(15, 6))\n    for i, r in enumerate(g.itertuples()):\n        color = '#d62728' if r.close >= r.open else '#2ca02c'\n        ax.vlines(i, r.low, r.high, color=color, linewidth=1)\n        ax.bar(i, max(abs(r.close-r.open), 0.001), bottom=min(r.open,r.close), width=0.65, color=color, alpha=0.75)\n    step = max(1, len(g)//10)\n    ax.set_xticks(x[::step]); ax.set_xticklabels(g['trade_time'].dt.strftime('%Y-%m-%d').iloc[::step], rotation=45, ha='right')\n    ax.set_title(f'{code} 日 K 线'); ax.set_ylabel('价格'); plt.show()\nfor code in daily['code'].unique(): plot_candles(daily, code)\n\n# 3 柱状图\nphase_table.T.plot(kind='bar', figsize=(12, 5), title='不同交易阶段的数据量'); plt.xlabel('交易阶段'); plt.ylabel('行数'); plt.show()\ncontinuous.groupby(['code', continuous['trade_time'].dt.date]).size().unstack(0).plot(kind='bar', figsize=(16,5), width=0.85, title='连续交易阶段每日快照数量'); plt.xlabel('日期'); plt.ylabel('快照行数'); plt.show()\n\n# 4 多折线图\nprice_wide = daily.pivot(index='trade_time', columns='code', values='close')\nprice_wide.plot(figsize=(14,6), linewidth=1.8, title='多 ETF 日收盘价走势'); plt.xlabel('日期'); plt.ylabel('收盘价'); plt.show()\n(price_wide / price_wide.ffill().bfill().iloc[0]).plot(figsize=(14,6), linewidth=1.8, title='多 ETF 归一化价格走势，首日=1'); plt.xlabel('日期'); plt.ylabel('归一化价格'); plt.show()

## 图 5-9：折柱图、阈值折线图、布林带、收益率直方图、累计收益率

In [ ]:
# 5 折柱图：价格 + 成交量\nfor code, g in daily.groupby('code'):\n    fig, ax1 = plt.subplots(figsize=(15,6)); ax2 = ax1.twinx()\n    ax1.plot(g['trade_time'], g['close'], color='#1f77b4', linewidth=1.8); ax2.bar(g['trade_time'], g['volume'], color='#888888', alpha=0.28)\n    ax1.set_title(f'{code} 价格趋势 + 成交量'); ax1.set_ylabel('价格'); ax2.set_ylabel('累计成交量'); plt.show()\n\n# 6 RSI 阈值折线图\ndef add_rsi(g, window=14):\n    g = g.sort_values('trade_time').copy(); d = g['close'].diff()\n    gain = d.clip(lower=0).rolling(window).mean(); loss = (-d.clip(upper=0)).rolling(window).mean()\n    g['rsi'] = 100 - 100 / (1 + gain / loss); return g\ndaily_rsi = daily.groupby('code', group_keys=False).apply(add_rsi)\nfor code, g in daily_rsi.groupby('code'):\n    plt.figure(figsize=(14,4)); plt.plot(g['trade_time'], g['rsi'], marker='o', linewidth=1.5)\n    plt.axhline(70, color='red', linestyle='--', label='超买 70'); plt.axhline(30, color='green', linestyle='--', label='超卖 30')\n    plt.ylim(0,100); plt.title(f'{code} RSI 阈值折线图'); plt.xlabel('日期'); plt.ylabel('RSI'); plt.legend(); plt.show()\n\n# 7 布林带\nfor code, g in daily.groupby('code'):\n    g = g.sort_values('trade_time').copy(); mid = g['close'].rolling(20).mean(); std = g['close'].rolling(20).std()\n    upper, lower = mid + 2*std, mid - 2*std\n    plt.figure(figsize=(14,6)); plt.plot(g['trade_time'], g['close'], label='close'); plt.plot(g['trade_time'], mid, label='20D MA')\n    plt.plot(g['trade_time'], upper, '--', label='upper'); plt.plot(g['trade_time'], lower, '--', label='lower'); plt.fill_between(g['trade_time'], lower, upper, alpha=0.12)\n    plt.title(f'{code} 布林带通道'); plt.xlabel('日期'); plt.ylabel('价格'); plt.legend(); plt.show()\n\n# 8 日收益率直方图\nfor code, g in daily.groupby('code'):\n    r = g['return'].dropna(); plt.figure(figsize=(10,5)); plt.hist(r, bins=30, alpha=0.75, edgecolor='white')\n    plt.axvline(r.mean(), color='red', linestyle='--', label=f'mean={r.mean():.6f}'); plt.title(f'{code} 日收益率直方图'); plt.xlabel('日收益率'); plt.ylabel('频数'); plt.legend(); plt.show()\n\n# 9 累计收益率折线图\ndaily.pivot(index='trade_time', columns='code', values='cum_return').plot(figsize=(14,6), linewidth=1.8, title='ETF 累计收益率')\nplt.axhline(0, color='black', linewidth=0.8); plt.xlabel('日期'); plt.ylabel('累计收益率'); plt.show()

## 保存 NaN 标记版\n\n这一步保存轻量清洗结果：保留 NaN，不填补，不先筛掉交易阶段。

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'a_nan_marked'\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\nsave_cols = [c for c in etf_all.columns if c != 'date']\nfor code, g in etf_all.groupby('code'):\n    short = code.replace('.SH', '')\n    start = g['trade_time'].min().strftime('%Y%m%d'); end = g['trade_time'].max().strftime('%Y%m%d')\n    out = OUTPUT_DIR / f'{short}_{start}_{end}_nan_marked.CSV'\n    g[save_cols].to_csv(out, index=False, encoding='utf-8-sig')\n    print(out)